# 06 — 1D-CNN Classification with 5-Fold Stratified Cross Validation

This notebook trains and evaluates a **1D Convolutional Neural Network (1D-CNN)** for binary
classification of NIR corn spectra (High-Protein vs Low-Protein) using
**5-Fold Stratified Cross Validation**.

The original `06_1d_cnn_training.ipynb` trained the model once on a fixed 80/20 split
(64 training, 16 test samples). With only 80 total samples, that single split is highly
sensitive to which samples happen to land in the test set — a one-sample shift changes
accuracy by ±6%. By training **five separate CNNs** — each on a different 64-sample fold —
and averaging the performance metrics, we obtain a more statistically reliable estimate
of model generalisation.

## Section 1 — Imports and Setup

We import all required libraries, add the project root to `sys.path` so that the `src/`
modules are importable, and fix every random seed to ensure full reproducibility across
runs. Setting seeds in Python's built-in `random`, NumPy, and TensorFlow ensures that
weight initialisation, data shuffling, and augmentation all produce the same outputs
each time the notebook is executed.

In [ ]:
import sys
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Allow imports from the project root (src/ directory)
sys.path.append('../..')

from src.augmentor import interpolation_augment
from src.cnn_trainer import (
    build_1d_cnn,
    reshape_for_cnn,
    plot_training_history,
    plot_cnn_confusion_matrix,
)

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

print("All libraries imported successfully.")
print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")

## Section 2 — Load Full Dataset

We load the Savitzky–Golay preprocessed spectra (`X_preprocessed.npy`) and their
corresponding binary protein labels (`y_labels.npy`).

**All 80 samples are loaded and kept intact at this stage.** No train/test split is
performed here. The cross-validation loop in Section 4 is solely responsible for
partitioning the data — this ensures no bias is introduced by a manual split made
before cross-validation begins.

In [ ]:
X = np.load('../../data/processed/X_preprocessed.npy')
y = np.load('../../data/processed/y_labels.npy')

print(f"X shape : {X.shape}  (samples x wavelengths)")
print(f"y shape : {y.shape}")
print()
print("Class distribution:")
unique, counts = np.unique(y, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = "High-Protein (1)" if cls == 1 else "Low-Protein  (0)"
    print(f"  Class {int(cls)} — {label}: {cnt} samples")
print(f"\nTotal samples: {len(y)}")

## Section 3 — Define 5-Fold Stratified Cross Validation

`StratifiedKFold` divides the 80 samples into 5 folds of 16 samples each while
**preserving the class ratio** in every fold. With only 80 samples, a non-stratified
split could accidentally produce a test fold where one protein class is
over-represented, which would bias the accuracy and F1-score for that fold.
Stratification guarantees that each fold is a representative mini-version of
the full dataset.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("5-Fold Stratified Cross Validation configured.")
print(f"  n_splits     : {skf.n_splits}")
print(f"  shuffle      : {skf.shuffle}")
print(f"  random_state : {skf.random_state}")
print(f"\nEach fold: ~{len(y) // skf.n_splits} test samples, "
      f"~{len(y) - len(y) // skf.n_splits} training samples")

## Section 4 — 1D-CNN 5-Fold Cross Validation

The following loop trains a **completely fresh 1D-CNN** for each of the five folds.
All preprocessing steps occur strictly within the loop so that no information from
any test fold can influence the training process:

1. **Augmentation** — `interpolation_augment` expands the 64 training spectra to
   2,000 synthetic samples by linearly interpolating between randomly paired
   same-class spectra (Li et al., 2025). The 16 test spectra are never augmented.
2. **Scaling** — `StandardScaler` is *fitted exclusively on the 2,000 augmented
   training samples* and then used to transform both sets. Fitting on the test
   fold would constitute data leakage.
3. **Reshape** — `reshape_for_cnn` adds the channel dimension required by
   Conv1D: `(n_samples, 700)` → `(n_samples, 700, 1)`.
4. **Training** — Up to 100 epochs, batch size 64, with 10 % of the training
   data held as an internal validation set. `EarlyStopping` (patience = 20)
   monitors `val_loss` and automatically restores the best weights.
5. **Best model tracking** — The fold that produces the **highest F1-score**
   on its test fold is retained as the final best model.

In [ ]:
os.makedirs('../../saved_models/revised', exist_ok=True)

# ── Storage for per-fold results ──────────────────────────────────────────────
cnn_fold_metrics = []
cnn_best_model   = None
cnn_best_f1      = 0.0
cnn_all_y_true   = []
cnn_all_y_pred   = []

# ── 5-Fold Cross Validation Loop ──────────────────────────────────────────────
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    print(f"\n{'='*56}")
    print(f"  Fold {fold + 1} / {skf.n_splits}")
    print(f"{'='*56}")

    # ── 1. Split ──────────────────────────────────────────────────────────────
    X_train_fold = X[train_idx]
    X_test_fold  = X[test_idx]
    y_train_fold = y[train_idx]
    y_test_fold  = y[test_idx]

    print(f"  Train size : {X_train_fold.shape[0]} samples")
    print(f"  Test size  : {X_test_fold.shape[0]} samples")

    # ── 2. Augment training fold only ─────────────────────────────────────────
    X_aug, y_aug = interpolation_augment(
        X_train_fold, y_train_fold,
        target_total=2000,
        random_state=RANDOM_STATE,
    )
    print(f"  Augmented  : {X_aug.shape[0]} training samples after augmentation")

    # ── 3. Scale (fit on augmented training data only) ────────────────────────
    scaler         = StandardScaler()
    X_aug_scaled   = scaler.fit_transform(X_aug)
    X_test_scaled  = scaler.transform(X_test_fold)

    # ── 4. Reshape for Conv1D: (n_samples, 700) -> (n_samples, 700, 1) ────────
    X_train_cnn = reshape_for_cnn(X_aug_scaled)
    X_test_cnn  = reshape_for_cnn(X_test_scaled)

    # ── 5. Build a fresh CNN for this fold ────────────────────────────────────
    model = build_1d_cnn(input_length=700)

    ckpt_path = f'../../saved_models/revised/cnn_fold_{fold + 1}.keras'
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=20,
            restore_best_weights=True,
            verbose=0,
        ),
        ModelCheckpoint(
            filepath=ckpt_path,
            monitor='val_loss',
            save_best_only=True,
            verbose=0,
        ),
    ]

    # ── 6. Train ──────────────────────────────────────────────────────────────
    history = model.fit(
        X_train_cnn, y_aug,
        epochs=100,
        batch_size=64,
        validation_split=0.1,
        callbacks=callbacks,
        verbose=1,
    )
    print(f"Fold {fold + 1} stopped at epoch {len(history.history['loss'])}")

    # ── 7. Plot training history for this fold ────────────────────────────────
    plot_training_history(history, title=f'1D-CNN Training History — Fold {fold + 1}')
    plt.show()

    # ── 8. Predict on the untouched test fold ─────────────────────────────────
    y_prob = model.predict(X_test_cnn, verbose=0)[:, 0]
    y_pred = (y_prob >= 0.5).astype(int)

    # ── 9. Compute metrics ────────────────────────────────────────────────────
    acc  = accuracy_score(y_test_fold, y_pred)
    prec = precision_score(y_test_fold, y_pred, zero_division=0)
    rec  = recall_score(y_test_fold, y_pred, zero_division=0)
    f1   = f1_score(y_test_fold, y_pred, zero_division=0)

    print(f"Fold {fold + 1}: Accuracy={acc:.4f}  Precision={prec:.4f}  "
          f"Recall={rec:.4f}  F1={f1:.4f}")

    cnn_fold_metrics.append(
        {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    )
    cnn_all_y_true.append(y_test_fold)
    cnn_all_y_pred.append(y_pred)

    # ── 10. Track best model by F1-score ──────────────────────────────────────
    if f1 > cnn_best_f1:
        cnn_best_f1   = f1
        cnn_best_model = model
        print(f"  *** New best model (Fold {fold + 1}, F1 = {cnn_best_f1:.4f}) ***")

print(f"\nCross-validation complete.")
print(f"Best fold F1-score : {cnn_best_f1:.4f}")

## Section 5 — CNN Results Summary

We summarise performance across all five folds by computing the mean and standard
deviation of each metric. The **combined confusion matrix** is constructed by
concatenating the predictions from every test fold. Because each of the 80 samples
belongs to exactly one test fold, every sample appears in the combined matrix exactly
once — giving a complete and unbiased picture of model performance across the
full dataset without any data leakage.

In [ ]:
metrics_df = pd.DataFrame(cnn_fold_metrics)

mean_acc  = metrics_df['accuracy'].mean()
std_acc   = metrics_df['accuracy'].std()
mean_prec = metrics_df['precision'].mean()
std_prec  = metrics_df['precision'].std()
mean_rec  = metrics_df['recall'].mean()
std_rec   = metrics_df['recall'].std()
mean_f1   = metrics_df['f1'].mean()
std_f1    = metrics_df['f1'].std()

print("====================================================")
print("  1D-CNN 5-Fold Cross Validation Results")
print("====================================================")
print(f"  Accuracy  : {mean_acc:.4f} +/- {std_acc:.4f}")
print(f"  Precision : {mean_prec:.4f} +/- {std_prec:.4f}")
print(f"  Recall    : {mean_rec:.4f} +/- {std_rec:.4f}")
print(f"  F1-Score  : {mean_f1:.4f} +/- {std_f1:.4f}")
print("====================================================")

# ── Per-fold breakdown ────────────────────────────────────────────────────────
print("\nPer-fold breakdown:")
print(metrics_df.rename(columns={
    'accuracy': 'Accuracy', 'precision': 'Precision',
    'recall': 'Recall', 'f1': 'F1-Score',
}).to_string(index=True))

# ── Combined confusion matrix (all 80 samples, each exactly once) ─────────────
y_true_all = np.concatenate(cnn_all_y_true)
y_pred_all = np.concatenate(cnn_all_y_pred)

plot_cnn_confusion_matrix(y_true_all, y_pred_all)
plt.show()

## Section 6 — Save Best CNN Model

The CNN from the fold that achieved the **highest F1-score** is saved as the
final model. We use the **native Keras format** (`.keras`) rather than the
legacy HDF5 (`.h5`) format because:

- TensorFlow 2.12+ designates `.keras` as the recommended default serialisation format.
- The `.h5` path triggers a deprecation warning in recent TensorFlow versions.
- `.keras` files are self-contained and can be reloaded with `keras.models.load_model()`
  without any additional arguments.

In [ ]:
import os

os.makedirs('../../saved_models/revised', exist_ok=True)

cnn_best_model.save('../../saved_models/revised/1d_cnn_best.keras')
print(f"Best CNN model saved: saved_models/revised/1d_cnn_best.keras")
print(f"Best fold F1-score  : {cnn_best_f1:.4f}")

## Section 7 — Summary

| Item | Detail |
|---|---|
| **Evaluation method** | 5-Fold Stratified Cross Validation |
| **Each fold** | 64 training samples augmented to 2,000; 16 test samples kept as original unmodified spectra |
| **Preprocessing** | `StandardScaler` fitted *inside each fold* on augmented training data only — never on test data |
| **CNN architecture** | Conv1D(32, k=11) → BN → MaxPool → Conv1D(64, k=7) → BN → MaxPool → Dense(32) → Dropout(0.3) → Dense(1, sigmoid) |
| **Optimiser** | Adam (lr = 0.0001), loss = binary cross-entropy |
| **Training** | Up to 100 epochs, batch size 64, EarlyStopping patience = 20, restore_best_weights = True |
| **Metrics** | Accuracy, Precision, Recall, F1-Score — reported as mean +/- std across 5 folds |
| **Best model** | Fold with highest F1-score saved to `saved_models/revised/1d_cnn_best.keras` |
| **Next notebook** | `notebooks/revised/07_evaluation_metrics.ipynb` — head-to-head comparison of PLS-DA, SVM, and 1D-CNN |

### Key design decisions

- **Augmentation inside the CV loop** — applying `interpolation_augment` only to the
  training fold prevents synthetic spectra derived from test-set samples from leaking
  into training, which would inflate reported accuracy.
- **StandardScaler inside the CV loop** — fitting the scaler exclusively on augmented
  training data mirrors the real deployment scenario where scaling parameters must be
  learned from training data alone.
- **Native `.keras` format** — avoids the HDF5 deprecation warning introduced in
  TensorFlow 2.12 and is the recommended Keras 3 serialisation format.
- **5-Fold CV rationale** — with only 80 samples a single 80/20 split produces a test
  set of 16 samples where a single misclassification shifts accuracy by 6.25 %.
  Averaging over five non-overlapping test folds substantially reduces this variance
  and yields a more reliable generalisation estimate for the thesis report.